# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata as an object; access fields as attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, record sets, fields, and columns are uniquely referenced by their `@id` values, which you should always use for analysis and extraction.

In [ ]:
# List all available record sets (@id)
record_sets = dataset.metadata.recordSet
if record_sets:
    print("Available Record Sets (referenced by @id):")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}")
else:
    print("No record sets found in metadata.")

# Show available fields/columns in each record set
for rs in (record_sets if record_sets else []):
    print(f"\nDetails for Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    print(f"Fields (@id):")
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"  - {field['@id']}")
        elif isinstance(field, str):
            print(f"  - {field}")
    print("Columns (@id):")
    for col in columns:
        if isinstance(col, dict) and '@id' in col:
            print(f"  - {col['@id']}")
        elif isinstance(col, str):
            print(f"  - {col}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

List-of-record-sets example based on typical Croissant schema:

- Survey responses: e.g. `cr:SurveyRecordSet`
- Demographics: e.g. `cr:DemographicsRecordSet`
- Psychological scores: e.g. `cr:ScoresRecordSet`

If the dataset uses other `@id`s for record sets, use the values shown in the previous cell.

In [ ]:
# Example: Extract all record sets and load into DataFrames
# Replace these with actual @id values for record sets from the overview
record_set_ids = []
if dataset.metadata.recordSet:
    for rs in dataset.metadata.recordSet:
        record_set_ids.append(rs['@id'])

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"Sample columns for record set {example_rs_id}:\n", dataframes[example_rs_id].columns.tolist())
    print(dataframes[example_rs_id].head())
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All fields and columns should be referenced by their `@id` (for example, `cr:age`, `cr:GAD7_score`).

We'll select a numeric field, filter, normalize, and group data.

In [ ]:
# Choose a record set and fields for EDA
if dataframes:
    # Pick first record set with data
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # You must reference fields by their @id -- update field IDs as appropriate for your dataset
    # For demo: try ['cr:GAD7_score', 'cr:PHQ9_score', 'cr:age']
    numeric_field_candidates = [col for col in df.columns if 'score' in col or 'age' in col]
    if not numeric_field_candidates:
        numeric_field_candidates = df.select_dtypes(include='number').columns.tolist()

    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field '@id': {numeric_field}")
        # Filtering
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field, e.g. gender or education, referenced by its @id
        group_field_candidates = [col for col in df.columns if col.startswith('cr:') and (
            'sex' in col or 'gender' in col or 'education' in col or 'village' in col)]
        if not group_field_candidates:
            group_field_candidates = df.select_dtypes(include='object').columns.tolist()

        if group_field_candidates:
            group_field = group_field_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Options include histograms, boxplots, or scatter plots for numeric fields, using their `@id`.

All plots reference fields by their `@id` for reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    numeric_field_candidates = [col for col in df.columns if 'score' in col or 'age' in col]
    if not numeric_field_candidates:
        numeric_field_candidates = df.select_dtypes(include='number').columns.tolist()

    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field} (by @id)")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        # Boxplot by group field, if available
        group_field_candidates = [col for col in df.columns if col.startswith('cr:') and (
            'sex' in col or 'gender' in col or 'education' in col or 'village' in col)]
        if not group_field_candidates:
            group_field_candidates = df.select_dtypes(include='object').columns.tolist()

        if group_field_candidates:
            group_field = group_field_candidates[0]
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field} (fields referenced by @id)")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded a FAIR² mental health survey dataset from Kilifi County, Kenya using the Croissant schema.
- Referenced all entities (record sets, fields, columns) strictly by their `@id` value for reproducibility.
- Extracted and explored the structure and content of available record sets.
- Performed basic filtering, normalization, and grouping operations.
- Visualized data distributions and relationships using `@id` references.

Proceed to further analyses using the structured DataFrames and Croissant metadata for reproducible and FAIR-compliant research.